# 02 — Data Preparation (Bab 5)
**Tahap lifecycle:** [L] Learning Problem — Data Preparation

Urutan: pembersihan dan deduplikasi → feature engineering → splitting → *fit* transformasi **hanya pada data training** → class weight.
Keluaran: `../data/processed/{train,val,test}.csv` dan `../models/vocab_bilstm.json`. **Test set tidak boleh disentuh sampai tahap evaluasi (Bab 7).**

In [1]:
import os, re, json
import numpy as np
import pandas as pd
import emoji
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

SEED = 42
np.random.seed(SEED)
DATA_PATH = '../data/indonesian_ai_chatbots_reviews.csv'
OUT_DIR = '../data/processed'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs('../models', exist_ok=True)

raw = pd.read_csv(DATA_PATH)
print('Data mentah:', raw.shape)

Data mentah: (29198, 15)


## 5.1 Data Cleaning
1. Hapus teks berlabel **konflik** (teks sama, label beda → label ambigu).
2. Hapus duplikat teks (sisakan satu) agar teks yang sama tidak masuk train dan test sekaligus.
3. Buang kolom yang tidak dipakai, termasuk `score` (penentu label → leakage) dan `userName` (privasi).

In [2]:
log = [('data mentah', len(raw))]

conf = raw.groupby('content').sentiment.nunique()
conf_texts = set(conf[conf > 1].index)
df = raw[~raw.content.isin(conf_texts)].copy()
print(f'Teks berlabel konflik: {len(conf_texts)} (total {len(raw)-len(df):,} baris dihapus)')
df = df.drop_duplicates('content', keep='first')
log.append(('setelah hapus label konflik + duplikat teks', len(df)))

KEEP_COLS = ['content', 'sentiment', 'contentLength', 'appName']   # appName hanya untuk analisis, bukan fitur
df = df[KEEP_COLS].copy()
df.shape

Teks berlabel konflik: 260 (total 5,219 baris dihapus)


(23272, 4)

## 5.3 Pembersihan teks (stateless — aman dilakukan sebelum split)
| Langkah | Alasan |
|---|---|
| lowercasing | menyatukan "Bagus"/"BAGUS"/"bagus" |
| emoji → token teks | emoji membawa sentimen, jangan dibuang |
| karakter berulang → maks. 2 | "baguuus" → "bagus" |
| hapus tanda baca/simbol | mengurangi noise |
| normalisasi bahasa gaul (kamus) | menyatukan ejaan tidak baku |
| **stopword tidak dihapus** | negasi ("tidak", "bukan") menentukan polaritas |
| **stemming tidak dilakukan** | stemmer kurang andal untuk bahasa gaul |

In [3]:
SLANG = {'gak':'tidak','ga':'tidak','nggak':'tidak','ngga':'tidak','gk':'tidak','tdk':'tidak','engga':'tidak','enggak':'tidak',
 'bgt':'banget','bngt':'banget','yg':'yang','dgn':'dengan','udh':'sudah','udah':'sudah','sdh':'sudah','aja':'saja','sih':'','kok':'',
 'bgs':'bagus','jg':'juga','utk':'untuk','krn':'karena','karna':'karena','tp':'tapi','org':'orang','lg':'lagi','lgi':'lagi',
 'dr':'dari','sy':'saya','gue':'saya','gw':'saya','klo':'kalau','kalo':'kalau','jd':'jadi','bs':'bisa','bsa':'bisa','trs':'terus',
 'bnyk':'banyak','mksih':'terima kasih','makasih':'terima kasih','thx':'terima kasih','thanks':'terima kasih','apk':'aplikasi',
 'app':'aplikasi','ok':'oke','emg':'memang','sm':'sama','blm':'belum','hrs':'harus','jgn':'jangan','lemot':'lambat','mantul':'mantap'}
print('Jumlah entri kamus gaul:', len(SLANG))

def clean(text):
    t = text.lower()
    t = emoji.demojize(t, language='id', delimiters=(' emo_', ' '))
    t = re.sub(r'(.)\1{2,}', r'\1\1', t)          # karakter berulang
    t = re.sub(r'[^a-z0-9_\s]', ' ', t)              # tanda baca/simbol
    toks = [SLANG.get(w, w) for w in t.split()]
    return ' '.join(' '.join(toks).split())

for s in raw.content.sample(4, random_state=7):
    print('ASLI :', s); print('BERSIH:', clean(s)); print()

Jumlah entri kamus gaul: 52
ASLI : gak bisa langsung buat video disuruh langganan dulu, baru mau coba digunakan
BERSIH: tidak bisa langsung buat video disuruh langganan dulu baru mau coba digunakan

ASLI : keren
BERSIH: keren

ASLI : makin kesini makin oon
BERSIH: makin kesini makin oon

ASLI : AI terbaik yang pernah ada
BERSIH: ai terbaik yang pernah ada



In [4]:
df['clean'] = df.content.map(clean)

n_empty = int((df.clean == '').sum())
df = df[df.clean != ''].copy()
log.append((f'hapus {n_empty} ulasan kosong setelah dibersihkan', len(df)))

# deduplikasi ulang pada teks bersih (duplikat yang hanya beda huruf besar/kecil/tanda baca) + label konflik baru
n_dup2 = int(df.clean.duplicated().sum())
cg = df.groupby('clean').sentiment.nunique(); conf2 = set(cg[cg > 1].index)
df = df[~df.clean.isin(conf2)].drop_duplicates('clean', keep='first')
log.append((f'dedup ulang ({n_dup2} duplikat, {len(conf2)} teks konflik)', len(df)))

pd.DataFrame(log, columns=['langkah', 'jumlah_baris'])

,langkah,jumlah_baris
0,data mentah,29198
1,setelah hapus label konflik + duplikat teks,23272
2,hapus 43 ulasan kosong setelah dibersihkan,23229
3,"dedup ulang (507 duplikat, 66 teks konflik)",22656


## 5.2 Feature Engineering — satu fitur baru: `n_excl`
Jumlah tanda seru pada teks **asli**. Tanda seru menandai emosi tetapi hilang saat tanda baca dihapus. Dipakai XGBoost bersama `contentLength`.

In [5]:
df['n_excl'] = df.content.str.count('!')
df[['n_excl', 'contentLength']].describe().round(2)

,n_excl,contentLength
count,22656.00,22656.00
mean,0.13,104.68
std,0.91,114.11
min,0.00,1.00
25%,0.00,27.00
50%,0.00,60.00
75%,0.00,136.00
max,61.00,500.00


## 5.4 Data Splitting — 70/15/15, stratified, `random_state=42`

In [6]:
train, tmp = train_test_split(df, test_size=0.30, stratify=df.sentiment, random_state=SEED)
val, test  = train_test_split(tmp, test_size=0.50, stratify=tmp.sentiment, random_state=SEED)

print(f'Total: {len(df):,} | Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')
dist = pd.DataFrame({n: d.sentiment.value_counts() for n, d in [('train', train), ('val', val), ('test', test)]})
display(dist)
display((dist / dist.sum() * 100).round(2))

# Cek kebocoran: tidak boleh ada teks yang sama antar split
assert len(set(train.clean) & set(test.clean)) == 0
assert len(set(train.clean) & set(val.clean)) == 0
assert len(set(val.clean) & set(test.clean)) == 0
print('Tidak ada teks yang sama antar split ✔')

Total: 22,656 | Train: 15,859 | Val: 3,398 | Test: 3,399


,train,val,test
sentiment,,,
negatif,6833,1464,1465
positif,6432,1378,1379
netral,2594,556,555


,train,val,test
sentiment,,,
negatif,43.09,43.08,43.10
positif,40.56,40.55,40.57
netral,16.36,16.36,16.33


Tidak ada teks yang sama antar split ✔


## 5.5 Data Balancing — *class weight*, dihitung **hanya dari data training**
Validation dan test dibiarkan pada distribusi aslinya.

In [7]:
classes = sorted(train.sentiment.unique())
class_weight = {c: len(train) / (len(classes) * (train.sentiment == c).sum()) for c in classes}
print('Rasio kelas terbesar : terkecil (train) =', round(train.sentiment.value_counts().max() / train.sentiment.value_counts().min(), 2), ': 1')
print('class_weight:', {k: round(v, 4) for k, v in class_weight.items()})

Rasio kelas terbesar : terkecil (train) = 2.63 : 1
class_weight: {'negatif': np.float64(0.7736), 'netral': np.float64(2.0379), 'positif': np.float64(0.8219)}


## 5.3 (lanjutan) Representasi teks menjadi angka — di-*fit* hanya pada data training
Hanya didemonstrasikan di sini (bentuk dan ukuran); notebook model akan melakukan *fit* ulang di dalam pipeline masing-masing.

In [8]:
# (a) TF-IDF untuk Linear SVM dan XGBoost
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=20000, min_df=2, sublinear_tf=True)
X_train = tfidf.fit_transform(train.clean)      # FIT hanya di train
X_val, X_test = tfidf.transform(val.clean), tfidf.transform(test.clean)
print('TF-IDF  train/val/test:', X_train.shape, X_val.shape, X_test.shape)

# (b) Tokenisasi + padding untuk Bi-LSTM (vocab dari train saja)
MIN_FREQ, MAX_LEN = 2, 64
tok_counts = Counter(w for s in train.clean for w in s.split())
vocab = {'<PAD>': 0, '<OOV>': 1}
for w, n in tok_counts.most_common():
    if n >= MIN_FREQ: vocab[w] = len(vocab)

def to_seq(text, max_len=MAX_LEN):
    ids = [vocab.get(w, 1) for w in text.split()][:max_len]      # truncating di akhir
    return ids + [0] * (max_len - len(ids))                       # padding di akhir

lens = train.clean.str.split().str.len()
print(f'Kosakata train: {len(tok_counts):,} | dipakai (freq>={MIN_FREQ}): {len(vocab)-2:,}')
print(f'Panjang teks train: median={lens.median():.0f}, p95={lens.quantile(.95):.0f}, p99={lens.quantile(.99):.0f}, maks={lens.max()} -> MAX_LEN={MAX_LEN}')
seq_train = np.array([to_seq(s) for s in train.clean]); print('Sekuens train:', seq_train.shape)

# (c) Label encoding
LABEL2ID = {'negatif': 0, 'netral': 1, 'positif': 2}
print('Label:', LABEL2ID)

TF-IDF  train/val/test: (15859, 20000) (3398, 20000) (3399, 20000)
Kosakata train: 15,366 | dipakai (freq>=2): 7,555
Panjang teks train: median=10, p95=62, p99=83, maks=103 -> MAX_LEN=64
Sekuens train: (15859, 64)
Label: {'negatif': 0, 'netral': 1, 'positif': 2}


## Simpan hasil (dipakai notebook 03–05)

In [9]:
cols = ['clean', 'sentiment', 'n_excl', 'contentLength', 'appName']
for name, d in [('train', train), ('val', val), ('test', test)]:
    d[cols].to_csv(f'{OUT_DIR}/{name}.csv', index=False)
json.dump(vocab, open('../models/vocab_bilstm.json', 'w'), ensure_ascii=False)
json.dump({'SEED': SEED, 'MAX_LEN': MAX_LEN, 'MIN_FREQ': MIN_FREQ, 'class_weight': class_weight, 'label2id': LABEL2ID},
          open('../models/prep_config.json', 'w'), indent=1)
print('Tersimpan:', os.listdir(OUT_DIR))

Tersimpan: ['test.csv', 'train.csv', 'val.csv']
